<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:
from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt
import tifffile
import cv2
import torch
import torchvision.transforms.v2 as v2

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from sklearn.cluster import KMeans, HDBSCAN

from tqdm.notebook import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from utils import *

RuntimeError: You're likely running Python from the parent directory of the sam2 repository (i.e. the directory where https://github.com/facebookresearch/sam2 is cloned into). This is not supported since the `sam2` Python package could be shadowed by the repository name (the repository is also named `sam2` and contains the Python package in `sam2/sam2`). Please run Python from another directory (e.g. from the repo dir rather than its parent dir, or from your home directory) after installing SAM 2.

In [ ]:
sam2_checkpoint = "checkpoints/sam2.1_hiera_large.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"

sam2_model = build_sam2(model_cfg, sam2_checkpoint, device=device)

predictor = SAM2ImagePredictor(sam2_model)

encoder = predictor.model.image_encoder
encoder.eval()

print("Model loaded on", device)

In [ ]:
imgs = tifffile.imread("../data/DP/max_projection.tif")

In [ ]:
    # Custom transform to repeat single channel 3 times
class RepeatChannels:
    def __call__(self, x):
        # x shape: (CYX)
        # print("shape recieved:", x.shape)
        x = x.repeat(3, 1, 1)  # shape (H, W, 3)
        # print("shape returned:", x.shape)
        return x


transforms = v2.Compose([
    v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),         # PIL -> Tensor (C,H,W)
    # RepeatChannels(), 
    v2.Resize((256, 256), interpolation=v2.InterpolationMode.NEAREST),
    v2.ConvertImageDtype(torch.float32),
])


In [ ]:
num_images = imgs.shape[0]
channel_predictions = []

for channel in range(4):
    predictions = []

    # Extract single channel and repeat 3 times for encoder
    one_channel_image = np.expand_dims(imgs[..., channel], axis=-1)  # (N,H,W,1)
    one_channel_image = np.repeat(one_channel_image, 3, axis=-1)      # (N,H,W,3)

    for image_idx in tqdm(range(num_images)):
    # for image_idx in tqdm(range(10)):
        image = one_channel_image[image_idx]  # shape (H,W,3)

        # Transform and move to device
        input_tensor = transforms(image).unsqueeze(0).to(device)  # (1,3,H,W)
        encoder_output = encoder(input_tensor)

        # Extract features
        l1_features = encoder_output["backbone_fpn"][0][0].detach().cpu().numpy()  # (C,H_feat,W_feat)
        C_feat, H_feat, W_feat = l1_features.shape
        features = l1_features.reshape(C_feat, -1).T  # (H_feat*W_feat, C_feat)

        # KMeans clustering
        kmeans = KMeans(n_clusters=2, init='k-means++', n_init=10, max_iter=500, random_state=0)
        kmeans.fit(features)
        preds = kmeans.predict(features).reshape(H_feat, W_feat)

        predictions.append(preds)

    # Stack predictions for this channel
    channel_predictions.append(np.stack(predictions, axis=0))  # shape: (N,H_feat,W_feat)

# Stack all channels along last dimension
predictions = np.stack(channel_predictions, axis=-1)  # shape: (N,H_feat,W_feat,4)
print("Predictions shape:", predictions.shape)


In [ ]:
plot_input_and_predictions(
    imgs[0],
    predictions[0],
    plot_channels_composite
)


### Adjusting for connected components

In [ ]:
predictions[...,3] = keep_intersecting_portions(predictions)

In [ ]:
plot_input_and_predictions(
    imgs[0],
    predictions[0],
    plot_channels_composite
)
